In [1]:
#!/usr/bin/env python3
"""
ingest_ratios_ttm_secuencial_auditado.py

Ingesta SECUENCIAL del endpoint ratios-ttm (FMP)
+ auditoría completa en infraestructura.update_logs
"""

import os
import time
import requests
import logging
import subprocess
from datetime import datetime, date
from dotenv import load_dotenv
import psycopg2

# ---------------------------
# ENV
# ---------------------------
load_dotenv()

POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST     = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT", "5432")
API_KEY           = os.getenv("FMP_API_KEY")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD, API_KEY]):
    raise EnvironmentError("Faltan variables de entorno")

# ---------------------------
# Logging
# ---------------------------
LOG_DIR  = "output_ingest"
os.makedirs(LOG_DIR, exist_ok=True)
LOG_FILE = f"{LOG_DIR}/ingest_ratios_ttm_{date.today().isoformat()}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler()
    ]
)

logging.info("=== START ingest_ratios_ttm_secuencial ===")

# ---------------------------
# Keep Mac awake
# ---------------------------
try:
    caffeinate = subprocess.Popen(["caffeinate"])
except Exception:
    caffeinate = None

# ---------------------------
# DB helpers
# ---------------------------
def get_conn():
    return psycopg2.connect(
        dbname=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD,
        host=POSTGRES_HOST,
        port=POSTGRES_PORT
    )

def log_db(conn, ticker, status, message):
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO infraestructura.update_logs
            (schema_name, table_name, ticker, status, message)
            VALUES (%s, %s, %s, %s, %s)
        """, (
            'api_raw',
            'multifactor_ratios_ttm',
            ticker,
            status,
            message
        ))
        conn.commit()

# ---------------------------
# Fetch tickers
# CAMBIO 1: ahora desde universos.stock_opciones_2026
# ---------------------------
def obtener_tickers(conn):
    with conn.cursor() as cur:
        cur.execute("SELECT ticker FROM universos.stock_opciones_2026")
        return [r[0] for r in cur.fetchall()]

# ---------------------------
# API
# CAMBIO 2: nuevo endpoint /stable/ratios-ttm
# ---------------------------
def fetch_ratios_ttm(ticker):
    url = f"https://financialmodelingprep.com/stable/ratios-ttm?symbol={ticker}&apikey={API_KEY}"
    r = requests.get(url, timeout=20)

    if r.status_code != 200:
        return None, f"HTTP_{r.status_code}"

    data = r.json()
    if not data:
        return None, "NO_DATA"

    return data[0], "OK"

# ---------------------------
# Insert
# ---------------------------
INSERT_SQL = """
INSERT INTO api_raw.multifactor_ratios_ttm (
    ticker, fecha_de_consulta,
    grossProfitMarginTTM, operatingProfitMarginTTM, netProfitMarginTTM,
    returnOnEquityTTM,
    priceEarningsRatioTTM, priceEarningsToGrowthRatioTTM,
    priceToSalesRatioTTM, priceToFreeCashFlowsRatioTTM,
    currentRatioTTM, quickRatioTTM,
    cashFlowToDebtRatioTTM, debtRatioTTM, debtEquityRatioTTM,
    interestCoverageTTM,
    freeCashFlowPerShareTTM, payoutRatioTTM
)
VALUES (
    %(ticker)s, %(fecha)s,
    %(grossProfitMarginTTM)s, %(operatingProfitMarginTTM)s, %(netProfitMarginTTM)s,
    %(returnOnEquityTTM)s,
    %(priceEarningsRatioTTM)s, %(priceEarningsToGrowthRatioTTM)s,
    %(priceToSalesRatioTTM)s, %(priceToFreeCashFlowsRatioTTM)s,
    %(currentRatioTTM)s, %(quickRatioTTM)s,
    %(cashFlowToDebtRatioTTM)s, %(debtRatioTTM)s, %(debtEquityRatioTTM)s,
    %(interestCoverageTTM)s,
    %(freeCashFlowPerShareTTM)s, %(payoutRatioTTM)s
)
ON CONFLICT (ticker, fecha_de_consulta)
DO UPDATE SET
    grossProfitMarginTTM          = EXCLUDED.grossProfitMarginTTM,
    operatingProfitMarginTTM      = EXCLUDED.operatingProfitMarginTTM,
    netProfitMarginTTM            = EXCLUDED.netProfitMarginTTM,
    returnOnEquityTTM             = EXCLUDED.returnOnEquityTTM,
    priceEarningsRatioTTM         = EXCLUDED.priceEarningsRatioTTM,
    priceEarningsToGrowthRatioTTM = EXCLUDED.priceEarningsToGrowthRatioTTM,
    priceToSalesRatioTTM          = EXCLUDED.priceToSalesRatioTTM,
    priceToFreeCashFlowsRatioTTM  = EXCLUDED.priceToFreeCashFlowsRatioTTM,
    currentRatioTTM               = EXCLUDED.currentRatioTTM,
    quickRatioTTM                 = EXCLUDED.quickRatioTTM,
    cashFlowToDebtRatioTTM        = EXCLUDED.cashFlowToDebtRatioTTM,
    debtRatioTTM                  = EXCLUDED.debtRatioTTM,
    debtEquityRatioTTM            = EXCLUDED.debtEquityRatioTTM,
    interestCoverageTTM           = EXCLUDED.interestCoverageTTM,
    freeCashFlowPerShareTTM       = EXCLUDED.freeCashFlowPerShareTTM,
    payoutRatioTTM                = EXCLUDED.payoutRatioTTM,
    updated_at                    = CURRENT_TIMESTAMP;
"""

# ---------------------------
# MAIN
# ---------------------------
def main():
    conn    = get_conn()
    tickers = obtener_tickers(conn)
    total   = len(tickers)

    logging.info(f"Tickers a procesar: {total}")

    ok, fail = 0, 0

    for i, ticker in enumerate(tickers, 1):
        try:
            data, status = fetch_ratios_ttm(ticker)

            if not data:
                log_db(conn, ticker, status, "Fetch failed")
                fail += 1
                continue

            # --------------------------------------------------
            # CAMBIO 3: mapeo explícito de métricas
            # Renombradas → nuevo nombre en la API
            # Eliminadas  → None (se inserta NULL en la tabla)
            # --------------------------------------------------
            payload = {
                "ticker"                        : ticker,
                "fecha"                         : date.today(),

                # OK — mismo nombre
                "grossProfitMarginTTM"          : data.get("grossProfitMarginTTM"),
                "operatingProfitMarginTTM"      : data.get("operatingProfitMarginTTM"),
                "netProfitMarginTTM"            : data.get("netProfitMarginTTM"),
                "currentRatioTTM"               : data.get("currentRatioTTM"),
                "quickRatioTTM"                 : data.get("quickRatioTTM"),
                "priceToSalesRatioTTM"          : data.get("priceToSalesRatioTTM"),
                "freeCashFlowPerShareTTM"       : data.get("freeCashFlowPerShareTTM"),

                # RENOMBRADAS → nuevo nombre en API
                "priceEarningsRatioTTM"         : data.get("priceToEarningsRatioTTM"),
                "priceEarningsToGrowthRatioTTM" : data.get("priceToEarningsGrowthRatioTTM"),
                "priceToFreeCashFlowsRatioTTM"  : data.get("priceToFreeCashFlowRatioTTM"),
                "debtEquityRatioTTM"            : data.get("debtToEquityRatioTTM"),
                "interestCoverageTTM"           : data.get("interestCoverageRatioTTM"),

                # ELIMINADA DE LA API → NULL
                "cashFlowToDebtRatioTTM"        : None,

                # ELIMINADAS DE LA API → NULL
                "returnOnEquityTTM"             : None,
                "debtRatioTTM"                  : None,
                "payoutRatioTTM"                : None,
            }

            with conn.cursor() as cur:
                cur.execute(INSERT_SQL, payload)
                conn.commit()

            log_db(conn, ticker, "success", "Inserted ratios_ttm")
            ok += 1

        except Exception as e:
            conn.rollback()
            log_db(conn, ticker, "exception", str(e))
            fail += 1
            logging.warning(f"{ticker} error: {e}")

        if i % 100 == 0:
            logging.info(f"Procesados {i}/{total}")

        time.sleep(0.25)

    conn.close()
    logging.info(f"FIN | OK={ok} | FAIL={fail}")

# ---------------------------
if __name__ == "__main__":
    try:
        main()
    finally:
        if caffeinate:
            caffeinate.terminate()

2026-04-07 10:55:17,841 | INFO | === START ingest_ratios_ttm_secuencial ===
2026-04-07 10:55:17,913 | INFO | Tickers a procesar: 3027
2026-04-07 10:56:54,876 | INFO | Procesados 100/3027
2026-04-07 10:58:33,977 | INFO | Procesados 200/3027
2026-04-07 11:00:14,696 | INFO | Procesados 300/3027
2026-04-07 11:01:55,063 | INFO | Procesados 400/3027
2026-04-07 11:03:34,717 | INFO | Procesados 500/3027
2026-04-07 11:05:13,011 | INFO | Procesados 600/3027
2026-04-07 11:06:52,786 | INFO | Procesados 700/3027
2026-04-07 11:08:29,955 | INFO | Procesados 800/3027
2026-04-07 11:10:07,721 | INFO | Procesados 900/3027
2026-04-07 11:11:45,845 | INFO | Procesados 1000/3027
2026-04-07 11:13:23,311 | INFO | Procesados 1100/3027
2026-04-07 11:14:58,982 | INFO | Procesados 1200/3027
2026-04-07 11:16:35,587 | INFO | Procesados 1300/3027
2026-04-07 11:18:32,030 | INFO | Procesados 1400/3027
2026-04-07 11:20:10,993 | INFO | Procesados 1500/3027
2026-04-07 11:21:47,236 | INFO | Procesados 1600/3027
2026-04-07 